In [36]:
import sys
import os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
from model_utils.plots import plot_results
from lstm import LSTM
from dataset import TimeSeriesDataset
import time
import itertools
import math


In [37]:
import os
import shutil
import stat
import time

def handle_remove_readonly(func, path, exc):
    # Callback to handle read-only files on Windows
    excvalue = exc[1]
    if func in (os.rmdir, os.remove, os.unlink) and excvalue.errno == 13: # EACCES
        os.chmod(path, stat.S_IWRITE)
        func(path)
    else:
        raise

# Clean up directories from previous runs
dirs_to_cleanup = ['best_models', 'grid_search_plots', 'training_logs', 'inference_logs']
for dir_path in dirs_to_cleanup:
    if os.path.exists(dir_path):
        # Retry a few times in case of transient locks
        for i in range(3):
            try:
                shutil.rmtree(dir_path, ignore_errors=False, onerror=handle_remove_readonly)
                print(f"Removed directory: {dir_path}")
                break
            except Exception as e:
                if i < 2:
                    time.sleep(1) # Wait a bit before retrying
                else:
                    print(f"Error removing {dir_path}: {e}")

Removed directory: best_models
Removed directory: grid_search_plots
Removed directory: training_logs
Removed directory: inference_logs


In [38]:
DATA_PATH = '../dataset/independent_items.feather'  # Adjust path if needed
print(f"Loading data from {DATA_PATH}...")
# Use pd.read_feather instead of feather.read_table to get a DataFrame directly
df = pd.read_feather(DATA_PATH)

NUM_ITEMS = 100
df = df[df['item_id'].isin(df['item_id'].unique()[:NUM_ITEMS])]

Loading data from ../dataset/independent_items.feather...


In [39]:
import holidays

# -----------------------------------------------------------------------------
# DATA LOADING & PREPROCESSING
# -----------------------------------------------------------------------------


# Define constants
DATE_COL = 'date'
TARGET_COL = 'value'  # Assuming 'value' is the target variable (sales at the end of the day)


# Handle DATE_COL (ensure it is a column and not in the index)
if DATE_COL in df.index.names:
    if DATE_COL in df.columns:
        # If it's in both, drop the index version to avoid "cannot insert" error
        df = df.reset_index(drop=True)
    else:
        # If it's only in the index, move it to a column
        df = df.reset_index()

# If DATE_COL is NOT in columns (and wasn't in index), we have a problem, but assuming it exists somewhere.
# Just to be safe, if we still have a complex index, reset it.
if df.index.name == DATE_COL:
     df = df.reset_index(drop=True)
     
# Fallback: simple reset to ensure RangeIndex 0..N
df = df.reset_index(drop=True)

# Ensure date is datetime and sort
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL)
df = df.reset_index(drop=True) # Final clean reset

# Prepare target variable
# Using 'value' as the target (sales at the end of the day)
y = df[TARGET_COL]
print(len(y))

# -----------------------------------------------------------------------------
# DATA SPLITTING
# -----------------------------------------------------------------------------
# Define split sizes
train_size = 455
val_size = 154
forecast_horizon = 152

# ensure day 2022-09-24 is the first day of test set
df = df.sort_values([DATE_COL, 'item_id', 'store_id']).reset_index(drop=True)


# -----------------------------------------------------------------------------
# SORT
# -----------------------------------------------------------------------------
df = df.sort_values([DATE_COL, "item_id", "store_id"]).reset_index(drop=True)

# -----------------------------------------------------------------------------
# BASIC CALENDAR PARTS
# -----------------------------------------------------------------------------
df["day_of_week"]   = df[DATE_COL].dt.dayofweek.astype(int)         # 0=Mon
df["day_of_month"]  = df[DATE_COL].dt.day.astype(int)
df["month"]         = df[DATE_COL].dt.month.astype(int)
df["moy"]           = (df["month"] - 1).astype(int)
df["quarter"]       = df[DATE_COL].dt.quarter.astype(int)
df["doy"]           = (df[DATE_COL].dt.dayofyear - 1).astype(int)
df["week_of_year"]  = df[DATE_COL].dt.isocalendar().week.astype(int)
df["year"]          = df[DATE_COL].dt.year.astype(int)

# weekend
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["is_monday"] = (df["day_of_week"] == 0).astype(int)
df["is_friday"] = (df["day_of_week"] == 4).astype(int)


# month / quarter boundaries
df["is_month_start"]   = df[DATE_COL].dt.is_month_start.astype(int)
df["is_month_end"]     = df[DATE_COL].dt.is_month_end.astype(int)
df["is_quarter_start"] = df[DATE_COL].dt.is_quarter_start.astype(int)
df["is_quarter_end"]   = df[DATE_COL].dt.is_quarter_end.astype(int)

# optional: week of month
df["week_of_month"] = ((df["day_of_month"] - 1) // 7 + 1).astype(int)

# -----------------------------------------------------------------------------
# CYCLICAL ENCODINGS
# -----------------------------------------------------------------------------
df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

df["month_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12)
df["month_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12)

df["doy_sin"] = np.sin(2 * np.pi * df["doy"] / 365.25)
df["doy_cos"] = np.cos(2 * np.pi * df["doy"] / 365.25)

# -----------------------------------------------------------------------------
# US HOLIDAYS
# -----------------------------------------------------------------------------
us_holidays = holidays.US()
holiday_dates = pd.to_datetime(sorted(us_holidays.keys()))

df["is_holiday"] = df[DATE_COL].isin(holiday_dates).astype(int)

# named holidays
df["is_christmas"] = (
    (df[DATE_COL].dt.month == 12) & (df[DATE_COL].dt.day == 25)
).astype(int)

df["is_thanksgiving"] = df[DATE_COL].apply(
    lambda x: 1 if us_holidays.get(x) == "Thanksgiving Day" else 0
)

# Black Friday (very useful for retail)
thanksgiving_dates = pd.to_datetime(
    [d for d, name in us_holidays.items() if name == "Thanksgiving Day"]
)
black_friday_dates = thanksgiving_dates + pd.Timedelta(days=1)
df["is_black_friday"] = df[DATE_COL].isin(black_friday_dates).astype(int)

# Christmas Eve / New Year's Eve
df["is_christmas_eve"] = (
    (df[DATE_COL].dt.month == 12) & (df[DATE_COL].dt.day == 24)
).astype(int)

df["is_new_year_eve"] = (
    (df[DATE_COL].dt.month == 12) & (df[DATE_COL].dt.day == 31)
).astype(int)

# -----------------------------------------------------------------------------
# HOLIDAY PROXIMITY FEATURES
# -----------------------------------------------------------------------------
# "near holiday" windows often help more than holiday-day itself
for lag in [1, 2, 3, 7]:
    df[f"is_pre_holiday_{lag}"] = 0
    df[f"is_post_holiday_{lag}"] = 0

for h in holiday_dates:
    for lag in [1, 2, 3, 7]:
        df.loc[df[DATE_COL] == h - pd.Timedelta(days=lag), f"is_pre_holiday_{lag}"] = 1
        df.loc[df[DATE_COL] == h + pd.Timedelta(days=lag), f"is_post_holiday_{lag}"] = 1

# -----------------------------------------------------------------------------
# LONG WEEKEND / BRIDGE-DAY FEATURES
# -----------------------------------------------------------------------------
# Friday before holiday Monday, Monday after holiday weekend, etc.
df["is_monday"] = (df["day_of_week"] == 0).astype(int)
df["is_friday"] = (df["day_of_week"] == 4).astype(int)

df["is_bridge_day"] = 0
holiday_set = set(holiday_dates)

for i, d in enumerate(df[DATE_COL]):
    prev_day = d - pd.Timedelta(days=1)
    next_day = d + pd.Timedelta(days=1)
    # workday between holiday and weekend
    if (prev_day in holiday_set and d.dayofweek == 4) or (next_day in holiday_set and d.dayofweek == 0):
        df.at[i, "is_bridge_day"] = 1
# USE THE EXOGENOUS COLUMNS YOU CREATED
EXOG_COLS = ["day_of_week", "doy", "is_thanksgiving", "is_christmas","is_weekend"]
'''
EXOG_COLS = [
    # base
    "day_of_week", "day_of_month", "week_of_year", "week_of_month",
    "month", "quarter", "is_weekend",
    "is_month_start", "is_month_end", "is_quarter_start", "is_quarter_end",

    # special days of the week
    "is_monday", "is_friday",
    # holidays
    "is_holiday", "is_thanksgiving", "is_black_friday",
    "is_christmas", "is_christmas_eve", "is_new_year_eve",
    "is_pre_holiday_1", "is_pre_holiday_2", "is_pre_holiday_3", "is_pre_holiday_7",
    "is_post_holiday_1", "is_post_holiday_2", "is_post_holiday_3", "is_post_holiday_7",

    # boundary / behavior
    "is_bridge_day"
]
'''
# Increase lookback window to help it learn more than just the past 7 days 
# when feeding predictions back recursively
lookback_window = 7 # Need at least the size of largest lag to unscale cleanly


76011


In [40]:
df

,date,item_id,value,cat_label,sdep_label,dep_label,dmn_label,promo_type_FRPG,promo_value_FRPG,promo_type_GAS,...,is_new_year_eve,is_pre_holiday_1,is_post_holiday_1,is_pre_holiday_2,is_post_holiday_2,is_pre_holiday_3,is_post_holiday_3,is_pre_holiday_7,is_post_holiday_7,is_bridge_day
0,2021-01-23,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
1,2021-01-23,156,3,dairy chs,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
2,2021-01-23,260,6,juices drnks shelf stbl,pos subd grocery other,pos dept grocery,juice/aseptic/new age,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
3,2021-01-23,278,26,sour cream,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
4,2021-01-23,314,4,cottage chs,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76006,2023-02-22,959943,3,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
76007,2023-02-22,961844,6,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
76008,2023-02-22,962444,13,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
76009,2023-02-22,983754,4,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0


In [41]:
print(df[TARGET_COL].describe())

count    76011.000000
mean         9.349055
std         11.011958
min          0.000000
25%          3.000000
50%          6.000000
75%         13.000000
max        333.000000
Name: value, dtype: float64


# Grid Search Cell

In [42]:
import time
import itertools
import sys
import os
import importlib
sys.path.append(os.path.abspath('..'))

# Reload plot_results to pick up changes in model_utils.utils
import model_utils.plots
importlib.reload(model_utils.plots)
from model_utils.plots import plot_results


# Modify lstm_experiment to return timings and handle plotting internally
def lstm_experiment_grid(df, target, item_id, store_id, train_size=500, val_size=100, forecast_window=161, 
                   seq_length=30, epochs=100, batch_size=32, lr=0.001, dropout=0.0, hidden_size=32, num_layers=1, exog_cols=None,
                   patience=50, seed=42, loss_type='MSELoss', save_plot_path=None, use_scheduler=False):

    # Set seed for reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    
    test_start_idx = len(df) - forecast_window
    val_start_idx = test_start_idx - val_size
    train_start_idx = val_start_idx - train_size

    print(f"Calculated indices - Train Start: {train_start_idx}, Val Start: {val_start_idx}, Test Start: {test_start_idx}")
    # Safety check
    if train_start_idx < 0:
        # If dataset is too short, we just start from 0
        train_start_idx = 0
        print(f"Warning: Dataset shorter than requested split sizes. adjusting train_start to 0.")

    train_slice = slice(train_start_idx, val_start_idx)
    val_slice = slice(val_start_idx, test_start_idx)
    test_slice = slice(test_start_idx, None)
    
    # Extract Target
    train = df[target][train_slice].values
    val = df[target][val_slice].values
    test = df[target][test_slice].values
    
    # Scale Target
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train.reshape(-1, 1)).flatten()
    val_scaled = scaler.transform(val.reshape(-1, 1)).flatten()

    # Extract Exogenous Variables
    exog_train = None
    exog_val = None
    full_exog = None
    
    if exog_cols and len(exog_cols) > 0:
        exog_train = df[exog_cols][train_slice].values
        exog_val = df[exog_cols][val_slice].values
        exog_test = df[exog_cols][test_slice].values
        # Scale Exogenous Variables
        exog_scaler = MinMaxScaler()
        exog_train_scaled = exog_scaler.fit_transform(exog_train)
        exog_val_scaled = exog_scaler.transform(exog_val)
        exog_test_scaled = exog_scaler.transform(exog_test)
    else:
         exog_train_scaled = None
         exog_val_scaled = None
         exog_test_scaled = None
    # Input size = 1 (target) + number of exog features
    input_size = 1 + (len(exog_cols) if exog_cols and len(exog_cols) > 0 else 0)
    
    # -------------------------------------------------------------------------
    # 2. Datasets & Loaders
    # -------------------------------------------------------------------------
    # Pass exogenous data to TimeSeriesDataset
    train_dataset = TimeSeriesDataset(train_scaled, exog_train_scaled, seq_length)
    use_pin_memory = torch.cuda.is_available()
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, pin_memory=use_pin_memory)
    
    # Important: Pass exog_val to validation dataset too!
    val_dataset = TimeSeriesDataset(val_scaled, exog_val_scaled, seq_length)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, pin_memory=use_pin_memory)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, dropout=dropout).to(device)
    
    if loss_type == 'MSELoss':
        criterion = nn.MSELoss()
    elif loss_type == 'L1Loss':
        criterion = nn.L1Loss()
    elif loss_type == 'HuberLoss':
        criterion = nn.HuberLoss()
    else:
        raise ValueError(f"Unsupported loss_type: {loss_type}")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    if use_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=patience//3)

    criterion2 = nn.MSELoss()  # For validation loss calculation
    
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    
    # Directories
    model_dir = f'best_models/seed_{seed}/{loss_type}'
    os.makedirs(model_dir, exist_ok=True)
    best_model_path = f'{model_dir}/lstm_item{item_id}_store{store_id}.pth'
    
    log_dir = f'training_logs/seed_{seed}/{loss_type}'
    os.makedirs(log_dir, exist_ok=True)
    log_path = f'{log_dir}/lstm_item{item_id}_store{store_id}_log.csv'

    best_epoch = 0
    
    # -------------------------------------------------------------------------
    # 3. Training Loop
    # -------------------------------------------------------------------------
    start_train_time = time.time()
    
    with open(log_path, 'w') as log_file:
        log_file.write("Epoch,Batch,Batch_X,Batch_Y,Output,Loss\n")
        
        for epoch in range(epochs):
            model.train()
            epoch_loss = 0
            for batch_idx, (batch_x, batch_y) in enumerate(train_loader):
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                optimizer.zero_grad()
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item()
                
                # Convert tensors to lists (represented as strings) to log in CSV
                batch_x_str = str(batch_x.cpu().detach().numpy().tolist()).replace('"', "'")
                batch_y_str = str(batch_y.cpu().detach().numpy().tolist()).replace('"', "'")
                outputs_str = str(outputs.cpu().detach().numpy().tolist()).replace('"', "'")
                
                log_file.write(f'{epoch},{batch_idx},"{batch_x_str}","{batch_y_str}","{outputs_str}",{loss.item()}\n')
            
            avg_train_loss = epoch_loss / len(train_loader)
            train_losses.append(avg_train_loss)
            
            # Validation
            model.eval()
            all_outputs = []
            all_targets = []

            with torch.no_grad():
                for batch_idx, (batch_x, batch_y) in enumerate(val_loader):
                    batch_x = batch_x.to(device)
                    batch_y = batch_y.to(device)
                    
                    outputs = model(batch_x)
                    all_outputs.append(outputs)
                    all_targets.append(batch_y)

            all_outputs = torch.cat(all_outputs, dim=0).view(-1)
            all_targets = torch.cat(all_targets, dim=0).view(-1)
            
            val_loss = criterion2(all_outputs, all_targets).item()
            val_losses.append(val_loss)
            
            if use_scheduler:
                scheduler.step(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_epoch = epoch + 1
                patience_counter = 0  # Reset patience counter on improvement
                torch.save(model.state_dict(), best_model_path)

            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1} due to no improvement in validation loss for {patience} epochs.")
                break
            patience_counter +=1
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.6f} | Val Loss: {val_loss:.6f}")

    train_time = time.time() - start_train_time

    # -------------------------------------------------------------------------
    # 4. Inference (Recursive)
    # -------------------------------------------------------------------------
    start_inference_time = time.time()
    
    if os.path.exists(best_model_path):
        model.load_state_dict(torch.load(best_model_path))
    else:
        print("Warning: No model saved. Using current model in memory.")

    model.eval()
    
    # Get the start date of the test set
    test_start_date = df[DATE_COL].iloc[test_start_idx]
    print(f"Test set starts on: {test_start_date.date()}")

    # Use the last seq_length points from validation data as initial input
    current_seq = val_scaled[-seq_length:].tolist()
    if exog_cols and len(exog_cols) > 0:
        # Shift exog by 1 to match training dataloader
        current_exog_seq = exog_val_scaled[-seq_length + 1:].tolist() + [exog_test_scaled[0].tolist()]
        
    current_date_seq = df[DATE_COL].iloc[test_start_idx - seq_length : test_start_idx].dt.date.tolist()
    
    forecast = []
    
    # Setup inference log file
    inf_log_dir = f'inference_logs/seed_{seed}/{loss_type}'
    os.makedirs(inf_log_dir, exist_ok=True)
    inf_log_path = f'{inf_log_dir}/inference_item{item_id}_store{store_id}.csv'
    
    with open(inf_log_path, 'w') as inf_log_file:
        
        # Dynamically identify all rolling mean features
        rolling_features = [] # List of tuples: (index, window_size)
        if exog_cols:
            for idx, col in enumerate(exog_cols):
                if col.startswith("rolling_mean_"):
                    try:
                        window = int(col.split("_")[-1])
                        rolling_features.append((idx, window))
                    except ValueError:
                        pass
        
        # Write header that includes exogenous columns if available
        header_str = "Step,X,Predicted_Y_Scaled,Predicted_Y_Unscaled"
        if exog_cols and len(exog_cols) > 0:
            exog_cols_unscaled_str = ",".join([f"{col}_Unscaled" for col in exog_cols])
            exog_cols_scaled_str = ",".join([f"{col}_Scaled" for col in exog_cols])
            header_str += f",{exog_cols_unscaled_str},{exog_cols_scaled_str}"
        inf_log_file.write(header_str + "\n")
        
        with torch.no_grad():
            for step in range(forecast_window):
                # Build model input from current history
                if exog_cols and len(exog_cols) > 0:
                    current_seq_arr = np.array(current_seq).reshape(-1, 1)
                    current_exog_arr = np.array(current_exog_seq)
                    x_np = np.column_stack([current_seq_arr, current_exog_arr])
                else:
                    x_np = np.array(current_seq).reshape(-1, 1)

                x = torch.FloatTensor(x_np).unsqueeze(0).to(device)

                # Predict next value
                pred = model(x).cpu().numpy()[0, 0]
                forecast.append(pred)

                # Log the unscaled array snapshot for validation right after predicting
                x_str = str(x_np.tolist()).replace('"', "'")
                
                # Unscale the prediction so it's readable in the log
                pred_unscaled = scaler.inverse_transform([[pred]])[0, 0]
                
                # Attempt to get the unscaled current exogenous features used for this prediction
                if exog_cols and len(exog_cols) > 0:
                    # Inverse transform just the last exogenous vector so we can read the raw values in the log
                    last_exog_scaled = current_exog_arr[-1]
                    last_exog_raw = exog_scaler.inverse_transform(last_exog_scaled.reshape(1, -1))[0]
                    
                    last_exog_unscaled_str = ",".join([str(v) for v in last_exog_raw.tolist()])
                    last_exog_scaled_str = ",".join([str(v) for v in last_exog_scaled.tolist()])
                    inf_log_file.write(f'{step},"{x_str}",{pred},{pred_unscaled},{last_exog_unscaled_str},{last_exog_scaled_str}\n')
                else:
                    inf_log_file.write(f'{step},"{x_str}",{pred},{pred_unscaled}\n')


                y_date = df[DATE_COL].iloc[test_start_idx + step].date()
                if step % 10 == 0:
                    print(f"Step {step}: Predicting for Date: {y_date}")

                # Update target sequence with prediction
                current_seq = current_seq[1:] + [pred]
                current_date_seq = current_date_seq[1:] + [y_date]

                # Update exogenous sequence for the row being appended
                if exog_cols and len(exog_cols) > 0 and step + 1 < forecast_window:
                    # Use raw exog for the next predicted date
                    next_exog_raw = exog_test[step + 1].copy()

                    # Recompute rolling means dynamically from past values only
                    if len(rolling_features) > 0:
                        max_w = max([w for _, w in rolling_features])
                        # Convert current sequence back to original scale (only what we strictly need)
                        hist_unscaled = scaler.inverse_transform(np.array(current_seq[-max_w:]).reshape(-1, 1)).flatten()
                        
                        for idx, w in rolling_features:
                            window_values = hist_unscaled[-w:]
                            next_exog_raw[idx] = np.mean(window_values) if len(window_values) > 0 else 0.0

                    # Scale after overwriting the rolling features
                    next_exog_scaled = exog_scaler.transform(next_exog_raw.reshape(1, -1))[0]

                    # Slide exog window
                    current_exog_seq = current_exog_seq[1:] + [next_exog_scaled.tolist()]
                    
    # Inverse transform predictions
    forecast = scaler.inverse_transform(np.array(forecast).reshape(-1, 1)).flatten()
    print(f"Forecasted values {forecast[:5]} ...")
    inference_time = time.time() - start_inference_time
    
    # Metrics
    rmse = np.sqrt(mean_squared_error(test, forecast))
    mae = mean_absolute_error(test, forecast)
    bias = np.mean(forecast - test)
    score = 0.5 * rmse + 0.25 * mae + 0.25 * abs(bias)
    
    def POCID(y_test, y_pred):
        diff_original = y_test[1:] - y_test[:-1]
        diff_pred = y_pred[1:] - y_pred[:-1]
        is_positive = (diff_original * diff_pred) > 0
        return is_positive.sum() / len(is_positive) if len(is_positive) > 0 else 0.0

    pocid = POCID(test, forecast)
    if save_plot_path:
        train_index = df[DATE_COL][train_slice].values
        val_index = df[DATE_COL][val_slice].values
        test_index = df[DATE_COL][test_slice].values
        
        plot_results(train, val, test, forecast, train_index, val_index, test_index, 
                     train_losses, val_losses, target, 
                     title=f'LSTM Forecast (Seed={seed}, Loss={loss_type}, Item={item_id}, Store={store_id})',
                     save_path=save_plot_path,
                     rmse=rmse, mae=mae, bias=bias, score=score, pocid=pocid)
    
    return rmse, mae, train_time, inference_time, best_epoch

In [43]:
# Grid Search Parameters
seeds = [2024]
loss_functions = ['MSELoss']  
batch_size = 32
hidden_size = 32
num_layers = 1
dropout = 0.0
EPOCHS = 1000  # Ensure EPOCHS is defined
LEARNING_RATE = 0.001

# Filter for specific products if needed
target_products = [27]
#target_products = [101054,101125, 101126, 102689, 103633, 103672, 103737, 103776, 103781,103782]
if target_products:
    products = df[df['item_id'].isin(target_products)][['item_id', 'store_id']].drop_duplicates().values
else:
    # Get all unique products from the subset dataset
    products = df[['item_id', 'store_id']].drop_duplicates().values

# Prepare results storage
results = []
os.makedirs('grid_search_plots', exist_ok=True)


In [44]:
# check products series count and check for any missing dates
item_id, store_id = products[0]  # Just checking the first product for now
# Filter data for the specific product
df_product = df[(df['item_id'] == item_id) & (df['store_id'] == store_id)].copy()
df_product

,date,item_id,value,cat_label,sdep_label,dep_label,dmn_label,promo_type_FRPG,promo_value_FRPG,promo_type_GAS,...,is_new_year_eve,is_pre_holiday_1,is_post_holiday_1,is_pre_holiday_2,is_post_holiday_2,is_pre_holiday_3,is_post_holiday_3,is_pre_holiday_7,is_post_holiday_7,is_bridge_day
0,2021-01-23,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
100,2021-01-24,27,14,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
200,2021-01-25,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
300,2021-01-26,27,5,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
400,2021-01-27,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75541,2023-02-18,27,15,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
75638,2023-02-19,27,10,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
75735,2023-02-20,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
75830,2023-02-21,27,3,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0


In [45]:


# Prepare results storage
results = []
os.makedirs('grid_search_plots', exist_ok=True)

# Run Grid Search
print(f"Starting Grid Search with {len(seeds)} seeds, {len(loss_functions)} loss functions and {len(products)} products...")

for seed in seeds:
    print(f"\n--- Processing Seed: {seed} ---")
    for loss_type in loss_functions:
        print(f"\n--- Processing Loss Type: {loss_type} ---")
        
        for item_id, store_id in products:
            print(f"Running: Seed={seed}, Loss={loss_type}, Item={item_id}, Store={store_id}")
            
            # Filter data for the specific product
            df_product = df[(df['item_id'] == item_id) & (df['store_id'] == store_id)].copy()
            
            # Handle DATE_COL (ensure it is a column and not in the index)
            if DATE_COL in df_product.index.names:
                if DATE_COL in df_product.columns:
                    # If it's in both, drop the index version to avoid "cannot insert" error
                    df_product = df_product.reset_index(drop=True)
                else:
                    # If it's only in the index, move it to a column
                    df_product = df_product.reset_index()

            # Fallback: simple reset to ensure RangeIndex 0..N
            df_product = df_product.reset_index(drop=True)

            df_product[DATE_COL] = pd.to_datetime(df_product[DATE_COL])
            df_product = df_product.sort_values(DATE_COL)
            df_product = df_product.reset_index(drop=True) # Final clean reset
            
            # Create directory for plots if it doesn't exist
            plot_dir = f'grid_search_plots/seed_{seed}/{loss_type}'
            os.makedirs(plot_dir, exist_ok=True)
            plot_filename = f'{plot_dir}/lstm_item{item_id}_store{store_id}.png'
            rmse, mae, train_time, infer_time, best_epoch = lstm_experiment_grid(
                df=df_product, 
                target=TARGET_COL, 
                item_id=item_id,
                store_id=store_id,
                train_size=train_size, 
                val_size=val_size,
                forecast_window=forecast_horizon, 
                seq_length=lookback_window,
                epochs=EPOCHS,  # Use the global EPOCHS setting (e.g., 1000)
                batch_size=batch_size, 
                lr=LEARNING_RATE,
                dropout=dropout,
                hidden_size=hidden_size,
                num_layers=1,
                patience=150,
                exog_cols=EXOG_COLS,
                seed=seed,
                loss_type=loss_type,
                save_plot_path=plot_filename,
                use_scheduler=True
            )
            
            results.append({
                'seed': seed,
                'loss_type': loss_type,
                'item_id': item_id,
                'store_id': store_id,
                'batch_size': batch_size,
                'hidden_size': hidden_size,
                'dropout': dropout,
                'rmse': rmse,
                'mae': mae,
                'train_time': train_time,
                'inference_time': infer_time,
                'best_epoch': best_epoch,
                'plot_path': plot_filename
            })
        

# Convert to DataFrame
results_df = pd.DataFrame(results)
results_df.to_csv('grid_search_results.csv', index=False)

Starting Grid Search with 1 seeds, 1 loss functions and 1 products...

--- Processing Seed: 2024 ---

--- Processing Loss Type: MSELoss ---
Running: Seed=2024, Loss=MSELoss, Item=27, Store=6269
Calculated indices - Train Start: 0, Val Start: 455, Test Start: 609
Epoch 10/1000 | Train Loss: 0.033660 | Val Loss: 0.032208
Epoch 20/1000 | Train Loss: 0.032641 | Val Loss: 0.031295
Epoch 30/1000 | Train Loss: 0.031942 | Val Loss: 0.030541
Epoch 40/1000 | Train Loss: 0.031435 | Val Loss: 0.029868
Epoch 50/1000 | Train Loss: 0.030985 | Val Loss: 0.029171
Epoch 60/1000 | Train Loss: 0.030579 | Val Loss: 0.029434
Epoch 70/1000 | Train Loss: 0.030344 | Val Loss: 0.029090
Epoch 80/1000 | Train Loss: 0.030099 | Val Loss: 0.028731
Epoch 90/1000 | Train Loss: 0.029848 | Val Loss: 0.028372
Epoch 100/1000 | Train Loss: 0.029599 | Val Loss: 0.028018
Epoch 110/1000 | Train Loss: 0.029353 | Val Loss: 0.028190
Epoch 120/1000 | Train Loss: 0.029233 | Val Loss: 0.028043
Epoch 130/1000 | Train Loss: 0.029118 

In [46]:
#df_inference= pd.read_csv('inference_logs/seed_2024/MSELoss/inference_item916110_store6269.csv')
#df_inference